# Spatial Disease Risk Modeling with INLA
### Small-Area Disease Mapping, Standardization, and Bayesian Shrinkage in R

---

### Small-Area Disease Mapping & Epidemiological Risk

In public health epidemiology, **disease mapping** is the primary spatial analytical method used to describe the geographic variation of disease incidence or mortality across small administrative areas (such as counties, municipalities, or health districts). Its fundamental objectives are:
1. **Surveillance & Etiology**: Formulate hypotheses about spatial risk factors and environmental or lifestyle hazards.
2. **Resource Allocation**: Direct targeted public health interventions and medical infrastructure to high-risk populations.
3. **True Risk Estimation**: Uncover the latent geographic pattern of disease risk from noisy, count-aggregated observational health registries.

#### 1. Standardization: Observed vs. Expected Cases
Raw disease counts $Y_i$ in county $i$ cannot be compared directly across regions because geographic units vary drastically in total population size, age structure, race, and sex composition. 

We account for demographic variation through **indirect standardization**. The population is partitioned into $J$ demographic strata (e.g., $J = 16$ cross-classifications of age, sex, and race). The stratum-specific reference rate across the entire study area is:
$$
r_j = \frac{\sum_{i=1}^n Y_{ij}}{\sum_{i=1}^n n_{ij}}, \quad j = 1, \dots, J
$$
The **expected number of cases** $E_i$ for area $i$ represents the number of cases expected if the area experienced the standard regional stratum-specific rates:
$$
E_i = \sum_{j=1}^J n_{ij} r_j
$$
The **Standardized Mortality (or Morbidity) Ratio (SMR)** is the classical ratio of observed to expected counts:
$$
\text{SMR}_i = \frac{Y_i}{E_i}
$$
* $\text{SMR}_i = 1$: Observed cases match regional demographic expectations.
* $\text{SMR}_i > 1$: Elevated mortality/morbidity relative to expectation.
* $\text{SMR}_i < 1$: Sub-baseline disease occurrence.

#### 2. The Small-Area Estimation Dilemma: Why Raw SMRs Fail
Despite its intuitive appeal, raw SMR has catastrophic inferential limitations in small-area studies:
* **Extreme Poisson Variance in Small Populations**: In sparsely populated rural counties where $E_i$ is very small (e.g., $E_i = 0.2$), a single observed case ($Y_i = 1$) produces an extreme $\text{SMR}_i = 5.0$, while $Y_i = 0$ yields $\text{SMR}_i = 0.0$.
* **Misleading Extreme Maps**: The highest and lowest raw SMR values consistently appear in the smallest, least reliable counties purely as artifacts of Poisson sampling noise.
* **No Spatial Information Borrowing**: Raw SMRs treat each county as completely independent of its neighbors, ignoring geographic continuity.

#### 3. Hierarchical Bayesian Smoothing (Besag-York-Mollié)
Hierarchical Bayesian disease mapping resolves small-area instability through **spatial shrinkage**. Observed counts are assumed Poisson-distributed with unknown relative risk $\theta_i$:
$$
Y_i \mid \theta_i \sim \text{Poisson}(E_i \theta_i), \quad i = 1, \dots, n
$$
The log-relative risk is modeled via a generalized linear mixed model with regional covariates and spatial random effects:
$$
\log(\theta_i) = \beta_0 + \sum_{k=1}^p \beta_k X_{ki} + u_i + v_i
$$
where:
* $\beta_0$ is the overall baseline log-risk.
* $X_{ki}$ are county-level risk covariates (e.g., adult smoking proportion).
* $u_i \sim \text{ICAR}(\tau_u)$ is an **Intrinsic Conditional Autoregressive (Besag)** spatial effect that borrows strength from adjacent neighboring counties.
* $v_i \sim \mathcal{N}(0, 1/\tau_v)$ is an unstructured Gaussian random effect capturing independent unmeasured local overdispersion.

#### 4. Bayesian Exceedance Probabilities
A major advantage of Bayesian disease mapping is the ability to compute exact **exceedance probabilities**:
$$
P(\theta_i > c \mid \mathbf{Y})
$$
which measures the posterior probability that relative risk in county $i$ exceeds a clinically or policy-meaningful threshold $c$ (e.g., $c = 1.2$, a $20\%$ elevated risk).

#### Data & Case Study
We analyze **lung cancer mortality across the 67 counties of Pennsylvania (1968-1980)** using the `pennLC` dataset from `SpatialEpi`, stratified by 16 age-sex-race demographic strata and adjusting for county-level adult smoking rates.


In [1]:
librarian::shelf(SpatialEpi, sf, spdep, INLA, tidyverse, patchwork)

# Configure INLA to avoid multi-process forking in Positron/Ark kernel
inla.setOption(num.threads = "1:1")

options(
  ark.plot.width = 10, ark.plot.height = 4.0,
  repr.plot.width = 10, repr.plot.height = 4.0
)


In [2]:
data(pennLC)

map <- st_as_sf(pennLC$spatial.polygon) |>
  mutate(county = purrr::map_chr(pennLC$spatial.polygon@polygons, \(p) p@ID))


In [3]:
d <- pennLC$data |>
  group_by(county) |>
  summarize(Y = sum(cases), .groups = "drop")

pennLC$data <- pennLC$data |>
  arrange(county, race, gender, age)

d <- d |>
  mutate(
    E = expected(
      population = pennLC$data$population,
      cases = pennLC$data$cases,
      n.strata = 16
    ),
    SMR = Y / E
  ) |>
  left_join(pennLC$smoking, by = "county")


In [4]:
map <- map |>
  left_join(d, by = "county")

head(st_drop_geometry(map))


     county    Y          E       SMR smoking
1     adams   55   69.62730 0.7899200   0.234
2 allegheny 1275 1182.42804 1.0782897   0.245
3 armstrong   49   67.61012 0.7247435   0.250
4    beaver  172  172.55806 0.9967660   0.276
5   bedford   37   44.19013 0.8372910   0.228
6     berks  308  300.70598 1.0242563   0.249

---
## 1. Classical Disease Surveillance: Expected Cases and Raw SMR

We begin by visualizing:
1. **Expected Cases ($E_i$)**: The demographic baseline representing county population density and age distribution.
2. **Raw SMR ($Y_i / E_i$)**: The un-smoothed empirical ratio of observed to expected deaths.


In [5]:
p_exp <- ggplot(map) +
  geom_sf(aes(fill = E), color = "white", linewidth = 0.2) +
  scale_fill_viridis_c(
    option = "cividis",
    name = "Expected (E)"
  ) +
  labs(
    title = "A. Demographically Expected Cases",
    subtitle = "Indirect standardization (16 age-sex-race strata)"
  ) +
  theme_void(base_size = 9.5) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(color = "grey40", hjust = 0.5, size = 8),
    legend.key.height = unit(0.8, "cm"),
    legend.key.width = unit(0.28, "cm"),
    plot.margin = margin(2, 4, 2, 4)
  )


In [6]:
p_smr_raw <- ggplot(map) +
  geom_sf(aes(fill = SMR), color = "white", linewidth = 0.2) +
  scale_fill_distiller(
    palette = "YlOrRd", direction = 1,
    name = "Raw SMR"
  ) +
  labs(
    title = "B. Raw SMR (Observed / Expected)",
    subtitle = "Unsmoothed county mortality ratio"
  ) +
  theme_void(base_size = 9.5) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(color = "grey40", hjust = 0.5, size = 8),
    legend.key.height = unit(0.8, "cm"),
    legend.key.width = unit(0.28, "cm"),
    plot.margin = margin(2, 4, 2, 4)
  )


In [7]:
options(ark.plot.width = 10.5, ark.plot.height = 3.8, repr.plot.width = 10.5, repr.plot.height = 3.8)
(p_exp | p_smr_raw) +
  plot_annotation(
    title = "Figure 10.1: Indirect Standardization and Empirical SMR (Pennsylvania Lung Cancer)",
    theme = theme(plot.title = element_text(face = "bold", hjust = 0.5, size = 12))
  )


### Notes on Figure 10.1
* **Expected Count Disparity (Panel A)**: Expected counts range from $E = 5.5$ cases in rural Forest County to $E = 1,775$ cases in urban Philadelphia. This $300$-fold population variation induces extreme sampling variance in rural SMRs.
* **Empirical Noise (Panel B)**: Raw SMR exhibits sharp geographic discontinuities. Contiguous rural counties fluctuate abruptly between $\text{SMR} = 0.65$ and $\text{SMR} = 1.35$, reflecting Poisson count noise rather than true localized epidemiology.
* **Next Step**: We stabilize these estimates using a Bayesian spatial hierarchical model that borrows strength across adjacent counties.


---
## 2. Hierarchical Bayesian Disease Risk Modeling

#### Model Formulation
We formulate the Poisson log-linear model:
$$
Y_i \mid \theta_i \sim \text{Poisson}(E_i \theta_i), \quad i = 1, \dots, 67
$$
$$
\log(\theta_i) = \beta_0 + \beta_1 \text{smoking}_i + u_i + v_i
$$
where:
* $\theta_i$ is the true underlying **Relative Risk** for county $i$.
* $E_i$ enters as a fixed **offset** ($\log(E_i)$ with regression coefficient constrained to 1).
* $\text{smoking}_i$ is the adult smoking proportion.
* $u_i \sim \text{Besag}$ is the scaled ICAR spatially structured random effect.
* $v_i \sim \mathcal{N}(0, 1/\tau_v)$ is the unstructured independent overdispersion effect.


In [8]:
nb <- poly2nb(map)
adj_file <- tempfile(fileext = ".adj")
nb2INLA(adj_file, nb)
g <- inla.read.graph(filename = adj_file)

map <- map |>
  mutate(
    re_u = row_number(),
    re_v = row_number()
  )


In [9]:
formula <- Y ~ smoking +
  f(re_u, model = "besag", graph = g, scale.model = TRUE) +
  f(re_v, model = "iid")

res <- inla(
  num.threads = "1:1",
  formula,
  family = "poisson",
  data = map,
  E = E,
  control.predictor = list(compute = TRUE),
  control.compute = list(return.marginals.predictor = TRUE, dic = TRUE)
)


In [10]:
res$summary.fixed


                 mean        sd  0.025quant   0.5quant 0.975quant      mode
(Intercept) -0.323503 0.1499387 -0.61956722 -0.3233551 -0.0284916 -0.323411
smoking      1.154558 0.6232297 -0.07704882  1.1559250  2.3795861  1.156189
                     kld
(Intercept) 3.791401e-08
smoking     3.795310e-08

In [11]:
threshold <- 1.20

map <- map |>
  mutate(
    RR = res$summary.fitted.values[, "mean"],        # Posterior Mean Relative Risk
    LL = res$summary.fitted.values[, "0.025quant"],  # 2.5% Lower Credible Bound
    UL = res$summary.fitted.values[, "0.975quant"],  # 97.5% Upper Credible Bound
    CI_width = UL - LL,                              # Posterior uncertainty bandwidth
    exc_prob = res$marginals.fitted.values |>
      purrr::map_dbl(\(marg) 1 - inla.pmarginal(q = threshold, marginal = marg))
  )

summary(map[, c("RR", "LL", "UL", "CI_width", "exc_prob")])


       RR               LL               UL            CI_width     
 Min.   :0.8460   Min.   :0.7456   Min.   :0.9322   Min.   :0.1078  
 1st Qu.:0.9170   1st Qu.:0.8126   1st Qu.:1.0163   1st Qu.:0.1768  
 Median :0.9469   Median :0.8477   Median :1.0602   Median :0.2028  
 Mean   :0.9552   Mean   :0.8559   Mean   :1.0600   Mean   :0.2041  
 3rd Qu.:0.9810   3rd Qu.:0.8916   3rd Qu.:1.0931   3rd Qu.:0.2313  
 Max.   :1.1526   Max.   :1.0958   Max.   :1.2278   Max.   :0.3376  
    exc_prob                  geometry 
 Min.   :0.000e+00   POLYGON      :67  
 1st Qu.:0.000e+00   epsg:NA      : 0  
 Median :5.124e-05   +proj=long...: 0  
 Mean   :2.130e-03                     
 3rd Qu.:3.537e-04                     
 Max.   :5.635e-02                     

### Covariate Estimates
* **Smoking Effect ($\beta_1 = 1.155$, $95\%\text{ CI: } [-0.077, 2.380]$)**:
  * Positively associated with lung cancer mortality.
  * For each $1\%$ ($0.01$) increase in county adult smoking proportion, lung cancer risk increases by $\approx \exp(0.01 \times 1.155) - 1 \approx +1.16\%$.
* **Bayesian Shrinkage in Action**:
  * In small rural counties where data are sparse, the posterior estimate of $\theta_i$ shrinks toward the neighboring local average, dampening random Poisson fluctuations.
  * In large urban counties (Philadelphia, Allegheny), the data dominate the prior, preserving localized precision.


In [12]:
common_lims <- range(c(map$SMR, map$RR))

p_smr_comp <- ggplot(map) +
  geom_sf(aes(fill = SMR), color = "white", linewidth = 0.2) +
  scale_fill_distiller(
    palette = "YlOrRd", direction = 1, limits = common_lims,
    name = "Risk"
  ) +
  labs(
    title = "A. Empirical SMR (Observed / Expected)",
    subtitle = "Noisy small-area estimates with extreme variance"
  ) +
  theme_void(base_size = 9.5) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(color = "grey40", hjust = 0.5, size = 8),
    legend.position = "none",
    plot.margin = margin(2, 4, 2, 4)
  )


In [13]:
p_rr_comp <- ggplot(map) +
  geom_sf(aes(fill = RR), color = "white", linewidth = 0.2) +
  scale_fill_distiller(
    palette = "YlOrRd", direction = 1, limits = common_lims,
    name = "Relative\nRisk"
  ) +
  labs(
    title = "B. Bayesian Smoothed Relative Risk (RR)",
    subtitle = "Model-based shrinkage borrowing strength across neighbors"
  ) +
  theme_void(base_size = 9.5) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(color = "grey40", hjust = 0.5, size = 8),
    legend.position = "right",
    legend.title = element_text(size = 8.5, face = "bold"),
    legend.key.height = unit(0.9, "cm"),
    legend.key.width = unit(0.28, "cm"),
    plot.margin = margin(2, 4, 2, 4)
  )


In [14]:
options(ark.plot.width = 10.8, ark.plot.height = 3.8, repr.plot.width = 10.8, repr.plot.height = 3.8)
(p_smr_comp | p_rr_comp) +
  plot_annotation(
    title = "Figure 10.2: Bayesian Spatial Shrinkage (Unified Risk Scale)",
    theme = theme(plot.title = element_text(face = "bold", hjust = 0.5, size = 12))
  )


### Notes on Figure 10.2
* **Noise Suppression**: The extreme artificial spikes visible in Panel A (raw SMR) are smoothed in Panel B into coherent, regional epidemiologic gradients.
* **Persistent True Hotspots**: Notice that elevated risk persists in western Pennsylvania (the Pittsburgh / Allegheny industrial belt) and northeastern mining valleys, confirming that high risk in these areas is systematic and structural, not a small-number statistical artifact.


In [15]:
p_exc <- ggplot(map) +
  geom_sf(aes(fill = exc_prob), color = "white", linewidth = 0.2) +
  scale_fill_viridis_c(
    option = "plasma",
    labels = scales::percent_format(accuracy = 1),
    name = "P(RR > 1.2)"
  ) +
  labs(
    title = "A. Exceedance Probability Map",
    subtitle = "Posterior probability that relative risk exceeds 1.20"
  ) +
  theme_void(base_size = 9.5) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(color = "grey40", hjust = 0.5, size = 8),
    legend.key.height = unit(0.9, "cm"),
    legend.key.width = unit(0.28, "cm"),
    plot.margin = margin(2, 4, 2, 4)
  )


In [16]:
philly_idx <- which(map$county == "philadelphia")
philly_marg <- inla.smarginal(res$marginals.fitted.values[[philly_idx]]) |> as_tibble()
philly_exc <- map$exc_prob[philly_idx]

p_dens <- ggplot(philly_marg, aes(x = x, y = y)) +
  geom_area(
    data = philly_marg |> filter(x >= threshold),
    aes(x = x, y = y),
    fill = "#e11d48", alpha = 0.6
  ) +
  geom_line(color = "#1e293b", linewidth = 0.8) +
  geom_vline(xintercept = 1.0, linetype = "dotted", color = "grey40", linewidth = 0.5) +
  geom_vline(xintercept = threshold, linetype = "dashed", color = "#e11d48", linewidth = 0.6) +
  annotate(
    "text", x = 1.0, y = max(philly_marg$y) * 0.9,
    label = "Baseline\n(RR = 1.0)",
    size = 2.8, color = "grey30", hjust = 1.1
  ) +
  annotate(
    "text", x = threshold, y = max(philly_marg$y) * 0.55,
    label = paste0("Threshold = 1.20\nP(RR > 1.2) = ", round(philly_exc * 100, 1), "%"),
    size = 2.8, color = "#be123c", fontface = "bold", hjust = -0.1
  ) +
  labs(
    title = "B. Posterior Distribution: Philadelphia County",
    subtitle = "Posterior marginal density of relative risk with exceedance tail",
    x = "Relative Risk (RR)",
    y = "Posterior Density"
  ) +
  theme_minimal(base_size = 9.5) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(color = "grey40", hjust = 0.5, size = 8),
    panel.grid.minor = element_blank(),
    plot.margin = margin(4, 8, 4, 8)
  )


In [17]:
options(ark.plot.width = 10.5, ark.plot.height = 4.0, repr.plot.width = 10.5, repr.plot.height = 4.0)
(p_exc | p_dens) +
  plot_annotation(
    title = "Figure 10.3: Spatial Exceedance Probabilities and Posterior Risk Distribution",
    theme = theme(plot.title = element_text(face = "bold", hjust = 0.5, size = 12))
  )


### Notes on Figure 10.3
* **Actionable Exceedance (Panel A)**: Instead of presenting point estimates alone, exceedance probabilities $P(\theta_i > 1.20 \mid \mathbf{Y})$ provide public health decision-makers with the exact statistical confidence that a county's excess mortality exceeds a critical threshold.
* **Posterior Marginal Distribution (Panel B)**:
  * For Philadelphia (the county with the highest estimated relative risk: Posterior Mean $= 1.15$, $95\%\text{ CrI: } [1.10, 1.23]$), the full posterior density shows significant elevation above statewide baseline ($\text{RR} = 1.0$).
  * However, the probability of exceeding the severe excess risk threshold of $1.20$ is only $5.6\%$ (represented by the shaded red upper tail).
  * This demonstrates the power of Bayesian disease mapping: point estimates alone ($\text{SMR}$ or $\hat{\theta}_i$) fail to convey uncertainty, whereas posterior marginal tails provide rigorous probabilistic evidence for policy interventions.
